# Chapitre 5 — Qui décode, qui normalise ? (CPU vs GPU)

**🎯 Objectif :** répondre par la **mesure** à une question de performance, pas de modélisation :
quand le GPU est rapide et les images grosses, faut-il laisser les workers CPU faire le z-score,
ou le déporter sur le GPU en une passe sur le batch entier ?
**⏱ Durée :** ~2–5 min sur GPU (12 runs chronométrés + amorçage du cache disque).

Cette annexe est un **aparté du chapitre 4** : elle réutilise ses données (les PNG du ch3) et
son architecture (ResNet-18 1 canal), mais elle ne cherche rien à apprendre — elle **chronomètre**.
Elle est séparée précisément pour ça : le ch4 enseigne l'entraînement, ce notebook mesure un
choix d'implémentation, et il coûte plusieurs minutes qu'on ne veut pas payer à chaque exécution
du ch3.

## Le problème

Un pas d'entraînement mêle du travail **CPU** (lire et **décoder** les images, les **normaliser**)
et du travail **GPU** (forward/backward). Le `DataLoader` prépare les batches avec des *workers*
CPU pendant que le GPU calcule. Mais si le GPU est rapide et les images grosses, les workers CPU
**n'arrivent pas à suivre** : le GPU attend les données et reste sous-utilisé — le pipeline est
*CPU/IO-bound*.

## L'astuce à évaluer

Laisser les workers CPU ne faire **que le décodage** (incompressible) et déplacer la
**normalisation z-score sur le GPU**, appliquée au **batch entier**. Aucune bibliothèque spéciale
n'est nécessaire : `x.mean(dim=(1,2,3))` et `x.std(dim=(1,2,3))` sont de simples opérations
tensorielles torch, qui s'exécutent là où vit le tenseur. Le GPU étant massivement parallèle,
normaliser tout un batch y est quasi gratuit, et le CPU est déchargé.

**Pourquoi le z-score et pas autre chose ?** Parce que c'est la seule transformation que le ch3
n'a pas pu figer sur disque : centrer-réduire produit des valeurs négatives, impossibles à stocker
dans un PNG uint8. Le crop, le resize et le flip sont faits une fois pour toutes — les déporter sur
GPU n'aurait aucun sens. Seul ce qui est **recalculé à chaque chargement** vaut la peine d'être
déplacé.

## Ce qu'on mesure

La normalisation seule serait trompeuse : ce qui compte, c'est le **débit du step complet**. On
mesure donc la **pipeline entière** (décodage des **vraies images** → z-score → forward/backward
ResNet-18), en comparant :

- **CPU-norm** : le z-score est fait dans les workers, image par image ;
- **GPU-norm** : les workers décodent seulement, le z-score se fait sur le batch une fois sur GPU.

Le tout **balayé sur plusieurs tailles de batch**, puis sur le **nombre de workers**.

> ⚠️ Le résultat **dépend de ta machine** (GPU, CPU, résolution, batch) : ce notebook n'affirme
> rien, il **mesure** et conclut d'après les chiffres. Sur un petit GPU ou un petit batch, le CPU
> peut très bien gagner. L'effet est **beaucoup plus marqué sur GMIC** (ch7), qui travaille en
> 2944×1920 — près de 22× plus de pixels par image : c'est là qu'on a mesuré le passage de
> **49 % à 91 %** d'utilisation GPU.

In [ ]:
import os, glob, time
import numpy as np
import cv2
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
from course_utils import preprocess_dir, pick_device, recommended_num_workers

DEVICE = pick_device(verbose=False)

# ────────────────────────────────────────────────────────────────────────────
DATASET = 'rsna_sample'      # <── LE CURSEUR, le MÊME qu'au ch3 et au ch4 :
#                                   'rsna_sample' (démo) ou 'rsna' (complet).
# ────────────────────────────────────────────────────────────────────────────
IMG_DIR = preprocess_dir(DATASET, 'cropped_images')   # même appel que la sortie du ch3

SIZE = 512                         # même résolution de travail qu'au ch4
N_WARMUP, N_STEPS = 3, 15          # steps ignorés (1er accès GPU anormalement lent) + steps mesurés
BATCH_SIZES = [2, 4, 8, 16]        # <- balaye ici (monte SIZE à 1024 pour stresser plus)
NUM_WORKERS = recommended_num_workers()

# sorted() : l'ordre que renvoie glob dépend du système de fichiers. Le figer garantit que les
# deux modes (CPU-norm / GPU-norm) décodent EXACTEMENT les mêmes fichiers, dans le même ordre —
# sinon on comparerait aussi, sans le savoir, deux sous-ensembles d'images différents.
_paths = sorted(glob.glob(os.path.join(IMG_DIR, '**', '*.png'), recursive=True))
assert _paths, (f"Aucune image dans {IMG_DIR} — exécute le ch1 (download) puis le ch3 "
                f"(preprocessing) avec DATASET = {DATASET!r}.")
print(f"{len(_paths)} PNG réels | device {DEVICE} | img {SIZE}² | workers {NUM_WORKERS}")

## Les trois briques

**`load`** fait le travail *commun* aux deux modes : décoder le PNG et le redimensionner. Pas de
normalisation ici — sinon il n'y aurait plus rien à déplacer.

**`normalize_image`** (CPU) et **`normalize_gpu`** (GPU) calculent **exactement le même z-score**,
à deux échelles différentes : une image `(H, W)` pour la version CPU, un batch `(N, 1, H, W)` pour
la version GPU. Le `dim=(1,2,3)` de la seconde moyenne sur canal+hauteur+largeur mais **pas** sur
la dimension 0 : chaque image du batch garde sa propre moyenne et son propre écart-type, comme
dans la version image par image.

**`build_bench_model`** reconstruit le ResNet-18 1-canal du ch4 en version minimale : `weights=None`
(un bench de *vitesse* n'a pas besoin des poids ImageNet, et ça évite tout accès réseau) et
`conv1` remplacée pour accepter 1 canal. L'explication complète de cette adaptation est au **ch4**.

In [ ]:
def load(filepath, size):
    """Décode un PNG et le redimensionne. AUCUNE normalisation : c'est la variable étudiée."""
    img = cv2.imread(filepath, cv2.IMREAD_UNCHANGED).astype(np.float32)
    return cv2.resize(img, (size, size))


def normalize_image(img):
    """z-score d'UNE image (numpy), tel qu'il tourne dans un worker CPU."""
    return (img - img.mean()) / max(img.std(), 1e-5)


def normalize_gpu(x):
    """Le MÊME z-score, mais sur un batch (N, 1, H, W) entier, là où vit le tenseur.

    dim=(1,2,3) = canal+hauteur+largeur, mais PAS la dimension 0 (le batch) : chaque image
    garde sa propre moyenne/écart-type, exactement comme la version image par image.
    """
    x = x.float()
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    # unbiased=False = écart-type "population" (÷N, pas ÷N-1) : la convention de numpy, donc
    # numériquement identique à normalize_image ci-dessus.
    s = x.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp(min=1e-5)
    return (x - m) / s


def build_bench_model():
    """ResNet-18 1 canal, poids aléatoires — version minimale du build_model() du ch3.

    weights=None : un bench de VITESSE n'a pas besoin des poids ImageNet (et évite tout accès
    réseau). Le forward/backward coûte exactement le même temps, entraîné ou pas.
    """
    m = resnet18(weights=None, num_classes=4)
    m.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    return m


# Vérification que les deux normalisations sont bien le même calcul (sinon on comparerait
# les vitesses de deux opérations différentes, ce qui n'aurait aucun sens).
_probe = load(_paths[0], SIZE)
_cpu = normalize_image(_probe)
_gpu = normalize_gpu(torch.from_numpy(_probe)[None, None]).numpy()[0, 0]
print('CPU-norm et GPU-norm identiques :', np.allclose(_cpu, _gpu, atol=1e-4),
      f'| écart max {np.abs(_cpu - _gpu).max():.2e}')

## Le harnais de mesure

`BenchDataset` porte **le seul interrupteur** de l'expérience : `cpu_norm`. À `True`, le worker
normalise avant de rendre l'image ; à `False`, il rend l'image brute et c'est `bench()` qui
appellera `normalize_gpu` sur le batch. Les deux modes renvoient du **float32 de même forme** :
le volume transféré vers le GPU est identique, donc l'écart mesuré vient bien de *qui* calcule
le z-score, et pas du coût de transfert.

Trois précautions rendent la mesure honnête, et chacune corrige un biais réel :

- **Warmup** : les `N_WARMUP` premiers steps ne sont pas chronométrés (allocation du contexte
  CUDA, cuDNN qui choisit ses algorithmes… le premier step est toujours anormalement lent).
- **`torch.cuda.synchronize()`** avant chaque lecture de l'horloge : les kernels CUDA sont
  **asynchrones**, donc sans ça `time.time()` mesurerait le temps que le CPU a mis à *envoyer*
  les ordres, pas celui que le GPU a mis à les *exécuter*.
- **Amorçage du cache disque** (cellule suivante) : sans lui, le run joué en premier lirait les
  PNG depuis le disque et le second depuis le cache du noyau — le second gagnerait pour une
  raison qui n'a rien à voir avec le z-score.

In [ ]:
class BenchDataset(Dataset):
    """Décode une vraie image ; si cpu_norm, applique AUSSI le z-score dans le worker (CPU)."""

    def __init__(self, n, cpu_norm):
        self.n = n
        self.cpu_norm = cpu_norm

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        # i % len(_paths) : on demande plus d'échantillons qu'il n'y a d'images réelles
        # (batch × 18 steps) — l'indice boucle donc sur la liste au lieu de la dépasser.
        img = load(_paths[i % len(_paths)], SIZE)
        if self.cpu_norm:
            img = normalize_image(img)
        # [None] ajoute l'axe CANAL : (H, W) -> (1, H, W). Le DataLoader empilera ensuite
        # batch_size de ces tenseurs en un (N, 1, H, W) — c'est LÀ que le batch apparaît.
        return torch.from_numpy(img)[None], i % 4      # label bidon : on chronomètre


def bench(batch_size, cpu_norm=True, n_workers=NUM_WORKERS):
    """Un run chronométré. cpu_norm=True -> z-score dans les workers ; False -> sur le batch GPU.

    Renvoie (ms/step, img/s) — ou (None, None) si le batch ne tient pas en mémoire GPU, pour que
    le sweep continue au lieu de s'arrêter net à la première taille trop grande.
    """
    # shuffle laissé à False (défaut) : l'ordre de lecture est le même d'un run à l'autre, donc
    # les deux modes subissent exactement le même coût d'accès disque.
    loader = DataLoader(BenchDataset(batch_size * (N_WARMUP + N_STEPS), cpu_norm),
                        batch_size=batch_size, num_workers=n_workers,
                        # pin_memory : mémoire hôte non paginable -> copie vers le GPU en DMA.
                        # Actif dans LES DEUX modes, donc sans effet sur la comparaison.
                        pin_memory=(DEVICE.type == 'cuda'))
    model = build_bench_model().to(DEVICE).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.CrossEntropyLoss()

    t_start = None
    try:
        for step, (x, y) in enumerate(loader):
            # y aussi sur DEVICE : la loss le compare à des logits déjà sur GPU — mélanger les
            # devices lèverait « expected all tensors to be on the same device ».
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if not cpu_norm:
                x = normalize_gpu(x)                      # le z-score que les workers n'ont PAS fait
            optimizer.zero_grad()
            loss_fn(model(x), y).backward()
            optimizer.step()
            if step == N_WARMUP - 1:                      # fin du warmup -> démarre le chrono
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                t_start = time.time()
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()                      # attend la fin réelle des kernels
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            torch.cuda.empty_cache()
            return None, None
        raise
    elapsed = (time.time() - t_start) / N_STEPS
    return elapsed * 1000, batch_size / elapsed           # ms/step, img/s


# Amorçage du cache disque : on décode une fois d'avance tous les fichiers que le sweep va lire,
# pour que TOUS les runs mesurés partent du même état (cache chaud) — cf. précaution 3 ci-dessus.
_n_prime = min(len(_paths), max(BATCH_SIZES) * (N_WARMUP + N_STEPS))
for p in _paths[:_n_prime]:
    load(p, SIZE)
print(f'Cache disque amorcé sur {_n_prime} images — les runs qui suivent sont comparables.')

## Mesure 1 — CPU-norm vs GPU-norm, par taille de batch

In [ ]:
_fmt = lambda ms, ips: f"{ms:8.0f} {ips:7.1f}" if ms else f"{'OOM / —':>16}"

print(f"{'batch':>5} | {'CPU-norm ms/step img/s':>22} | {'GPU-norm ms/step img/s':>22} | verdict")
print('-' * 78)
for bs in BATCH_SIZES:
    c_ms, c_ips = bench(bs, cpu_norm=True)
    # Sans GPU, « GPU-norm » normaliserait sur le même CPU : la comparaison n'aurait aucun sens.
    g_ms, g_ips = bench(bs, cpu_norm=False) if DEVICE.type == 'cuda' else (None, None)
    verdict = (f"GPU-norm {c_ms/g_ms:.2f}×" if g_ms < c_ms else f"CPU-norm {g_ms/c_ms:.2f}×") \
              if (c_ms and g_ms) else ("(pas de GPU)" if DEVICE.type != 'cuda' else "OOM")
    print(f"{bs:5d} | {_fmt(c_ms, c_ips)} | {_fmt(g_ms, g_ips)} | {verdict}")

print("\nOn mesure le STEP complet (décodage + z-score + forward/backward). Le gain GPU-norm")
print("n'apparaît que si le z-score CPU sature les workers ; sinon le CPU peut très bien gagner.")

## Mesure 2 — combien de workers, vraiment ?

`recommended_num_workers()` plafonne à **4** par **portabilité** (une valeur sûre partout), sans
mesurer si 4 est le bon chiffre *ici*. On le mesure : batch fixe, nombre de workers variable, et
on regarde où le débit **plafonne** — c'est le point où les cœurs sont saturés, au-delà duquel
ajouter des workers ne sert plus à rien.

On joue ce sweep en **GPU-norm** quand un GPU est là : les workers ne font alors *que* décoder,
ce qui isole le coût du décodage — le vrai goulot quand le pipeline est CPU/IO-bound.

In [ ]:
CPU_COUNT = os.cpu_count() or 1
_gpu_mode = (DEVICE.type == 'cuda')
_label = 'gpu-norm' if _gpu_mode else 'cpu-norm'
BS_FIXED = 16

print(f"{'workers':>7} | {'img/s (' + _label + f', batch={BS_FIXED})':>26}")
print('-' * 38)
for nw in sorted({0, 2, 4, CPU_COUNT}):        # 4 = plafond actuel de recommended_num_workers()
    _, ips = bench(BS_FIXED, cpu_norm=not _gpu_mode, n_workers=nw)
    print(f"{nw:7d} | {ips:26.1f}" if ips else f"{nw:7d} | {'OOM / —':>26}")

print(f"\n{CPU_COUNT} cœurs ici. Là où le débit plafonne, les cœurs sont saturés.")
print("recommended_num_workers() reste à 4 pour rester safe partout ; exporte COURSE_NUM_WORKERS=<n>")
print("pour imposer la valeur mesurée ici sur une machine puissante.")

## Comment lire ces chiffres

| Ce que tu observes | Ce que ça signifie | Quoi faire |
|---|---|---|
| GPU-norm gagne nettement, et le débit monte avec les workers | Pipeline **CPU/IO-bound** : les workers étaient le goulot | déporter le z-score sur GPU, et monter `COURSE_NUM_WORKERS` |
| Les deux modes sont à égalité | Le CPU suit le GPU à cette résolution | ne rien changer, la complexité ne se paierait pas |
| CPU-norm gagne (souvent à petit batch) | Le GPU est déjà le goulot, ou le batch est trop petit pour amortir le kernel | garder le z-score côté CPU |
| Le débit plafonne dès 2 workers | Le décodage n'est pas le goulot | inutile d'ajouter des cœurs |

**Le point à retenir pour la suite.** À 512×512 le verdict est souvent serré. À **2944×1920**
(GMIC, ch7) il ne l'est plus : ~22× plus de pixels par image, donc un décodage et un z-score
bien plus coûteux **par image**, face à un GPU qui, lui, ne ralentit pas d'autant. C'est le
régime CPU/IO-bound franc — et c'est là que déporter la normalisation a fait passer
l'utilisation GPU de **49 % à 91 %** sur le fine-tuning réel de ce projet.

Retour au **[chapitre 4](04_resnet18_breast_density.ipynb)**, ou suite avec le
**[chapitre 6](06_gmic_architecture.ipynb)** (architecture GMIC).